In [ ]:
import asyncio
from fastmcp import Client

client = Client("http://localhost:8000/mcp")

async def call_tool(name: str):
    async with client:
        result = await client.call_tool("get_test", {"location": name})
        print(result)
        return result


In [ ]:
data  = await call_tool("Ford")

In [ ]:
data.content[0].text

# MCP Loading Resources

In [ ]:
from langchain_mcp_adapters.client import MultiServerMCPClient

import aiofile.aio as aio
import base64
import pandas as pd

In [ ]:
f_path = "/home/user/Documents/Health Auto Export/health_metric_data/test.txt"

In [ ]:
with open(f_path, 'rb') as fp:
    data = fp.read()


In [ ]:
base64.b64encode(data).decode('utf-8')

In [ ]:

base64.b64encode(f_path)

In [ ]:
pd.read_json()

In [ ]:
pd.read_csv()

In [ ]:
mcp_server_settings = {
    "Ubuntu-OS-Filesystem": {
        "transport": "http",
        "url": "http://localhost:8000/mcp"
    }
}

In [ ]:
client = MultiServerMCPClient(mcp_server_settings)


In [ ]:

blobs = await client.get_resources("Ubuntu-OS-Filesystem")


# SANDBOX CLIENT

In [ ]:
import os, sys
f_path = "/home/user/gh/anubis-project/anubis-mcp-server-ubuntu"
sys.path.insert(0, f_path)
from src.client.client import load_directory_into_local_backend, create_local_backend

In [ ]:
from deepagents.backends import LocalShellBackend
from langchain_mcp_adapters.client import MultiServerMCPClient
import json

In [ ]:
mcp_client = MultiServerMCPClient({
    "Ubuntu-OS-Filesystem": {
        "transport": "http",
        "url": "http://localhost:8000/mcp"
    }
})

In [ ]:
tools = await mcp_client.get_tools()

In [ ]:
list_tool = next(tool for tool in tools if tool.name == "list_all_files")

In [ ]:
from src.server.app import HEALTH_DATA_ROOT
files = await list_tool.ainvoke({
    "directory": HEALTH_DATA_ROOT.__str__(),
    "recursive": True,
})

In [ ]:
files_list = json.loads(files[0]['text'])

In [ ]:
files_list

In [ ]:
async def mcp_tool(client, name: str, args: dict):
    tools = await client.get_tools()
    tool = next(tool for tool in tools if tool.name == name)
    return await tool.ainvoke(args)

In [ ]:
HEALTH_DATA_ROOT.__str__()

In [ ]:
files  = await mcp_tool(mcp_client, "list_all_files", {"directory": HEALTH_DATA_ROOT.__str__()})

# Start

In [ ]:
import os, sys, json, asyncio
from pathlib import Path
PROJECT = "/home/user/gh/anubis-project/anubis-mcp-server-ubuntu"
sys.path.insert(0, PROJECT)
from langchain_mcp_adapters.client import MultiServerMCPClient
from deepagents import create_deep_agent
from langgraph.checkpoint.memory import InMemorySaver
from src.server.app import HEALTH_DATA_ROOT
from src.client.client import (
    ANALYSIS_ROOT,
    LOCAL_SHELL_DATA_PROMPT,
    create_local_backend,
    load_directory_into_local_backend,
    make_agent_config,
    mcp_tool,
)

In [ ]:
mcp_client = MultiServerMCPClient({
    "Ubuntu-OS-Filesystem": {
        "transport": "http",
        "url": "http://localhost:8000/mcp",
    }
})
files_list = await mcp_tool(mcp_client, "list_all_files", {
    "directory": str(HEALTH_DATA_ROOT),
    "recursive": True,
})
len(files_list), files_list[:3]

In [ ]:
files_list[0]

In [ ]:
preview = await mcp_tool(mcp_client, "preview_data", {
    "file_path": files_list[0],
    "n_rows": 5,
})
preview["rows"][0]["data"][:2]


# Create Backend Sandbox

In [ ]:
backend = create_local_backend()
print(backend.execute("pwd && ls -la"))

In [ ]:
uploaded_paths = await load_directory_into_local_backend(
    mcp_client,
    str(HEALTH_DATA_ROOT),
    backend,
    data_dir="/data",
    # limit=5,
)
print(backend.ls("/data"))
uploaded_paths

In [ ]:
checkpointer = InMemorySaver()
config = make_agent_config(recursion_limit=100)

agent = create_deep_agent(
    model="gpt-5.4-nano",
    tools=[],
    backend=backend,
    checkpointer=checkpointer,
)

In [ ]:
input_message = {
    "role": "user",
    "content": (
        f"Health JSON exports are in /data/ ({len(uploaded_paths)} files). "
        f"{LOCAL_SHELL_DATA_PROMPT} "
        "Each file is a daily Apple Health Auto Export with nested metric records. "
        "Explore one file's structure, combine metrics across files, "
        "write a markdown report to /health_report.md, "
        "and save at least one visualization to /health_plot.png. "
        "Use pandas and matplotlib via execute()."
    ),
}


In [ ]:
uploaded_paths

In [ ]:
from pprint import pprint as pp

In [ ]:
pp(input_message['content'])

In [ ]:
async for event in agent.astream_events(
    {"messages": [input_message]},
    config,
    version="v2",
):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        chunk = event["data"].get("chunk")
        if chunk and chunk.content:
            print(chunk.content, end="", flush=True)
    elif kind == "on_tool_start":
        print(f"\n→ {event['name']}", flush=True)
    elif kind == "on_tool_end":
        output = event["data"].get("output")
        if output:
            preview = str(output)[:200]
            print(f"  {preview}{'…' if len(str(output)) > 200 else ''}", flush=True)

In [ ]:
from rich.markdown import Markdown

In [ ]:
backend.ls("/")

In [ ]:

from pathlib import Path
from src.client.client import ANALYSIS_ROOT
print("In backend:", (ANALYSIS_ROOT / "health_report.md").exists())
print("Stray copy:", Path("/tmp/health_report.md").exists())

In [ ]:
report = backend.read("/health_report.md")
if report.file_data:
    display(Markdown(report.file_data['content']))
else:
    print(report.error)


In [ ]:
import IPython

In [ ]:
IPython.display.Image('/tmp/health-analysis/health_plot.png')

In [ ]:
ANALYSIS_ROOT

In [ ]:

# On disk: ANALYSIS_ROOT / "health_report.md"
print((ANALYSIS_ROOT / "health_report.md").exists())

In [ ]:
async def ask_agent(question: str):
    msg = {"role": "user", "content": question}
    result = await agent.ainvoke({"messages": [msg]}, config)
    result["messages"][-1].pretty_print()
    return result

In [ ]:
await ask_agent('Am I in shape')

In [ ]:
await ask_agent("Which day had the most steps?")

In [ ]:
await ask_agent("Compare that to the flights climbed on the same day")

In [ ]:
await ask_agent("How many steps will be taken in the future if this trend continues?")

In [ ]:
await ask_agent("How many steps will be taken in the future if this trend continues? Please fit a linear model and forecast the steps within the next two weeks")

In [ ]:
await ask_agent("How many steps will be taken in the future if this trend continues? Please fit a linear model and forecast the steps within the next two weeks, and what do I need to do to increase my health with respect to the average number of daily steps a person like me takes? I'm a middle aged man average height and weight")

In [ ]:
await ask_agent("How many steps will be taken in the future if this trend continues? Please fit a linear model and forecast the steps within the next two weeks, and what do I need to do to increase my health with respect to the average number of daily steps a person like me takes? I'm a middle aged man average height and weight. please compute an average. please use the average to make a realistic estimage of the continuing trend. The model does not need to be linear.")

In [ ]:
await ask_agent("Compare the average daily steps of a man in their mid 30s with my data. Make suggestions if I am above or below this level of fitness with respect to this data")

In [ ]:
from src.client.client import cleanup_analysis

In [ ]:
await cleanup_analysis()

In [ ]:
backend.execute("pwd")